# Notebook 02 — Signal Analysis (RQ1)
**RQ1:** Which signal-level properties of IMU data differ systematically between elderly and young-adult subjects performing comparable activities?

Analyses:
- Acceleration magnitude peaks (SVM)
- Signal variability (std, IQR)
- Jerk (first derivative of acceleration)
- Dominant frequency and spectral energy (FFT)
- Autocorrelation (gait regularity proxy)
- Postural-transition duration


In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_ROOT = '/content/drive/MyDrive/fall-detection-generalization'
    %pip install -q numpy pandas scipy matplotlib seaborn tqdm scikit-learn
    sys.path.insert(0, REPO_ROOT)
else:
    import pathlib
    REPO_ROOT = str(pathlib.Path('..').resolve())

DATA_ROOT = REPO_ROOT + '/data/raw'

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import signal as sp_signal
from scipy.stats import mannwhitneyu, kruskal
from tqdm.notebook import tqdm

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

from src.data.sisfall import load_sisfall, SAMPLE_RATE as FS_SIS
from src.data.harmonize import compute_svm, compute_net_acceleration, resample_signal

RESULTS = Path(REPO_ROOT) / 'results'
RESULTS.mkdir(exist_ok=True)

In [ ]:
trials = load_sisfall(Path(DATA_ROOT) / 'sisfall')
print(f'Loaded {len(trials)} trials')

## 1. Per-trial signal features

In [ ]:
def trial_features(trial, fs=FS_SIS):
    """Compute scalar summary features for one trial."""
    acc = trial.signal[:, :3].astype(float)
    svm = compute_svm(acc)
    net = compute_net_acceleration(acc)
    jerk = np.diff(svm) * fs  # m/s² or g/s

    # Spectral
    freqs, psd = sp_signal.welch(svm, fs=fs, nperseg=min(256, len(svm)))
    dom_freq = freqs[np.argmax(psd)]
    spectral_energy = np.sum(psd)
    spectral_entropy = -np.sum((psd / psd.sum() + 1e-12) * np.log(psd / psd.sum() + 1e-12))

    # Autocorrelation regularity (stride regularity proxy for walking)
    normed = (svm - svm.mean()) / (svm.std() + 1e-8)
    autocorr = np.correlate(normed, normed, mode='full')
    autocorr = autocorr[len(autocorr) // 2:]
    autocorr /= autocorr[0]

    return {
        'subject_id':       trial.subject_id,
        'activity':         trial.activity,
        'is_fall':          trial.is_fall,
        'age_group':        trial.age_group,
        # Magnitude
        'svm_mean':         svm.mean(),
        'svm_std':          svm.std(),
        'svm_max':          svm.max(),
        'svm_iqr':          np.percentile(svm, 75) - np.percentile(svm, 25),
        'net_acc_max':      net.max(),
        # Jerk
        'jerk_max':         np.abs(jerk).max(),
        'jerk_std':         jerk.std(),
        # Spectral
        'dom_freq_hz':      dom_freq,
        'spectral_energy':  spectral_energy,
        'spectral_entropy': spectral_entropy,
        # Autocorrelation
        'autocorr_1st_peak': autocorr[1:].max() if len(autocorr) > 1 else 0.0,
    }

records = [trial_features(t) for t in tqdm(trials, desc='Extracting signal features')]
df_feat = pd.DataFrame(records)
df_feat.to_csv(RESULTS / 'signal_features.csv', index=False)
df_feat.head()

## 2. ADL-only comparison (young vs elderly, same activity type)

In [ ]:
df_adl = df_feat[~df_feat['is_fall']].copy()

feature_cols = ['svm_std', 'svm_max', 'net_acc_max', 'jerk_max',
                'dom_freq_hz', 'spectral_energy', 'autocorr_1st_peak']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, feat in enumerate(feature_cols):
    ax = axes[i]
    sns.boxplot(data=df_adl, x='age_group', y=feat,
                order=['young', 'elderly'], ax=ax,
                palette={'young': '#4c72b0', 'elderly': '#dd8452'})
    # Mann-Whitney U test
    young_vals   = df_adl.loc[df_adl['age_group'] == 'young',   feat].dropna()
    elderly_vals = df_adl.loc[df_adl['age_group'] == 'elderly', feat].dropna()
    if len(young_vals) > 0 and len(elderly_vals) > 0:
        stat, p = mannwhitneyu(young_vals, elderly_vals, alternative='two-sided')
        ax.set_title(f'{feat}\np={p:.3g}', fontsize=9)
    else:
        ax.set_title(feat, fontsize=9)

axes[-1].set_visible(False)
plt.suptitle('ADL signal features: young vs elderly (SisFall)', y=1.01)
plt.tight_layout()
plt.savefig(RESULTS / 'rq1_adl_feature_comparison.png', bbox_inches='tight')
plt.show()

## 3. Fall vs ADL overlap — do elderly ADLs look like young falls?

In [ ]:
df_feat['group'] = df_feat.apply(
    lambda r: f"{r['age_group']} {'fall' if r['is_fall'] else 'ADL'}", axis=1
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

palette = {
    'young fall':     '#d62728',
    'young ADL':      '#1f77b4',
    'elderly ADL':    '#ff7f0e',
    'elderly fall':   '#9467bd',
}

for ax, (x_feat, y_feat) in zip(axes, [
    ('svm_max', 'jerk_max'),
    ('net_acc_max', 'svm_std'),
]):
    for grp, grp_df in df_feat.groupby('group'):
        ax.scatter(grp_df[x_feat], grp_df[y_feat], label=grp,
                   alpha=0.5, s=20, color=palette.get(grp, 'grey'))
    ax.set_xlabel(x_feat)
    ax.set_ylabel(y_feat)
    ax.legend(fontsize=8)

plt.suptitle('Fall vs ADL overlap in signal feature space')
plt.tight_layout()
plt.savefig(RESULTS / 'rq1_fall_adl_overlap.png', bbox_inches='tight')
plt.show()

## 4. Average power spectra — young vs elderly ADL

In [ ]:
psd_records = {grp: [] for grp in ['young', 'elderly']}
adl_trials = [t for t in trials if not t.is_fall]

for trial in adl_trials:
    svm = compute_svm(trial.signal[:, :3].astype(float))
    freqs, psd = sp_signal.welch(svm, fs=FS_SIS, nperseg=min(256, len(svm)))
    psd_records[trial.age_group].append(psd)

fig, ax = plt.subplots(figsize=(10, 5))
for grp, color in [('young', '#1f77b4'), ('elderly', '#ff7f0e')]:
    psds = np.array(psd_records[grp])
    mean_psd = psds.mean(axis=0)
    std_psd  = psds.std(axis=0)
    ax.semilogy(freqs, mean_psd, label=f'{grp} (n={len(psds)})', color=color)
    ax.fill_between(freqs, mean_psd - std_psd, mean_psd + std_psd, alpha=0.2, color=color)

ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('PSD (g²/Hz)')
ax.set_title('Mean power spectral density — ADL trials (SisFall)')
ax.legend()
ax.set_xlim(0, 20)
plt.tight_layout()
plt.savefig(RESULTS / 'rq1_psd_comparison.png', bbox_inches='tight')
plt.show()

## 5. Summary statistics table (Table for thesis §6 / RQ1 answer)


In [ ]:
summary_rows = []
for feat in feature_cols:
    for grp in ['young', 'elderly']:
        vals = df_adl.loc[df_adl['age_group'] == grp, feat].dropna()
        summary_rows.append({
            'feature':   feat,
            'age_group': grp,
            'mean':      vals.mean(),
            'std':       vals.std(),
            'median':    vals.median(),
        })

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(RESULTS / 'rq1_summary_stats.csv', index=False)
print(df_summary.pivot(index='feature', columns='age_group').round(4).to_string())